*AI-generated draft (Claude, Anthropic) — for review. The picking UI and logic are version-controlled; the Scene-1 times you record are your own judgments.*

<span style="font-family: 'Courier New', monospace;">

# 32 · Pick Scene-1 times — CAMHDA301-2021 TRAINING set (v2 retrain)

Same picker as notebook 30, but for the **-2021 training frames** (different recordings than the validation ones — no leakage). For each contact sheet, find the **front-on Mushroom** tile and record its **`t=___s`** time; these frames are then pre-labeled and box-corrected to fine-tune the detector.

**Kernel:** `joseph-scaleworm-thesis`.

1. Run both cells. 2. Read the front-on tile's `t=___s`, type it, **Save & Next ▶**. 3. If no usable front-on view, **No usable view**. 4. Zoom to read labels; **◀ Prev** to revisit. Saves to `pick_sheet.csv`, resumable.

When done: `python scripts/make_prelabels.py datasets/scaleworm_v2/train_pick_2021/pick_sheet.csv datasets/scaleworm_v2/prelabels/2021`
</span>

In [1]:
%matplotlib widget
import datetime as dt
from pathlib import Path

import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

BASE = Path("/home/jovyan/scaleworm-student-lab/datasets/scaleworm_v2/train_pick_2021")
SHEET = BASE / "pick_sheet.csv"
SHEETS = BASE / "contact_sheets"

df = pd.read_csv(SHEET, dtype=str, keep_default_na=False)
pending = [
    i
    for i in df.index
    if df.at[i, "camera_unit"] == "CAMHDA301-2021"
    and (SHEETS / f"{df.at[i, 'frame_id']}.png").exists()
]
picked = sum(1 for i in pending if df.at[i, "scene1_time_s"].strip() != "")
print(f"{len(pending)} -2021 contact sheets  ({picked} already have a Scene-1 time).")
print("Run the next cell. For each sheet: read the t=..s label of the front-on")
print("Mushroom tile, type it in the box, Save & Next. Use 'No usable view' to skip.")

22 -2021 contact sheets  (0 already have a Scene-1 time).
Run the next cell. For each sheet: read the t=..s label of the front-on
Mushroom tile, type it in the box, Save & Next. Use 'No usable view' to skip.


In [2]:
class ScenePicker:
    """Record the Scene-1 (front-on Mushroom) time for each -2021 contact sheet."""

    def __init__(self, df, order):
        self.df = df
        self.order = order
        # open on the first row that is neither picked nor already skipped
        self.pos = next(
            (
                k
                for k, i in enumerate(order)
                if df.at[i, "scene1_time_s"].strip() == ""
                and df.at[i, "frame_ready"] != "no-scene1"
            ),
            0,
        )

        self.time = widgets.FloatText(
            value=0.0, step=1.0, description="t = ",
            layout=widgets.Layout(width="190px"),
        )
        self.status = widgets.HTML()
        self.msg = widgets.Output()

        def mk(desc, style=""):
            return widgets.Button(
                description=desc, button_style=style,
                layout=widgets.Layout(width="auto"),
            )

        self.b_prev = mk("◀ Prev")
        self.b_save = mk("Save & Next ▶", "success")
        self.b_skip = mk("No usable view", "warning")
        self.b_prev.on_click(lambda _: self._prev())
        self.b_save.on_click(lambda _: self._save())
        self.b_skip.on_click(lambda _: self._skip())

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 7))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_position = "right"

        controls = widgets.HBox([self.time, self.b_prev, self.b_save, self.b_skip])
        self.box = widgets.VBox([controls, self.status, self.fig.canvas, self.msg])
        self._load()

    def _fid(self):
        return self.df.at[self.order[self.pos], "frame_id"]

    def _load(self):
        fid = self._fid()
        self.ax.clear()
        self.ax.imshow(mpimg.imread(SHEETS / f"{fid}.png"))
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        self.ax.set_title(fid, fontsize=10)
        cur = self.df.at[self.order[self.pos], "scene1_time_s"].strip()
        self.time.value = float(cur) if cur else 0.0
        self.fig.canvas.draw_idle()
        self._status()

    def _status(self):
        row = self.df.loc[self.order[self.pos]]
        cur = row["scene1_time_s"].strip() or "—"
        note = f" &nbsp; <i>{row['notes']}</i>" if row["notes"] else ""
        self.status.value = (
            f"<b>Sheet {self.pos + 1}/{len(self.order)}</b> &nbsp; "
            f"{row['datetime_utc']} &nbsp; scene1_time_s: <b>{cur}</b>{note}"
        )

    def _write(self):
        self.df.to_csv(SHEET, index=False)

    def _save(self):
        i = self.order[self.pos]
        t = float(self.time.value)
        if t <= 0:
            self.msg.clear_output()
            with self.msg:
                print("⚠ Enter the tile's t=..s value (seconds) before saving.")
            return
        self.df.at[i, "scene1_time_s"] = f"{t:g}"
        if self.df.at[i, "frame_ready"] == "no-scene1":
            self.df.at[i, "frame_ready"] = "pending-contact-sheet"
        if self.df.at[i, "notes"] == "no usable Scene-1 view":
            self.df.at[i, "notes"] = ""
        self._write()
        self._advance()

    def _skip(self):
        i = self.order[self.pos]
        self.df.at[i, "scene1_time_s"] = ""
        self.df.at[i, "frame_ready"] = "no-scene1"
        self.df.at[i, "notes"] = "no usable Scene-1 view"
        self._write()
        self._advance()

    def _advance(self):
        self.msg.clear_output()
        if self.pos < len(self.order) - 1:
            self.pos += 1
            self._load()
        else:
            with self.msg:
                print("✅ All -2021 sheets reviewed. Next: "
                      "python scripts/extract_pending_frames.py")

    def _prev(self):
        if self.pos > 0:
            self.pos -= 1
            self._load()


app = ScenePicker(df, pending)
display(app.box)